# Cleaning and reshaping

## Summary
The pipeline retains 2,800 dates, creates 39,200 date-service rows, and preserves 27,007 observed counts and all nulls. The source checksum is recorded in the pipeline manifest.

## Context and methods
Exact duplicate rows may be removed and counted. Conflicting date rows and invalid dates/counts stop processing. No imputation, anomaly removal, interpolation or manual edits are applied. The long form lets the same service/mode dimensions work in SQL and Tableau.

### Assumptions
The schema is explicit: newly added services require a reviewed mapping change. Month and weekday features describe calendar dates; weekends are not a holiday proxy.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.config import RAW_PATH, OUTPUT_DIR, SERVICES, CORE_SERVICES
from src.data_cleaning import clean_data, quality_profile, reshape_data, validate_outputs
pd.set_option('display.max_rows', 16)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)


## Data
Inspect the source and service mapping.

In [2]:
raw = pd.read_csv(RAW_PATH)
display(pd.DataFrame([{'service_code': code, 'service': name, 'mode': mode} for code, (name, mode) in SERVICES.items()]))

,service_code,service,mode
0,bus_rkl,Rapid Bus KL,Bus
1,bus_rkn,Rapid Bus Kuantan,Bus
2,bus_rpn,Rapid Bus Penang,Bus
3,rail_lrt_ampang,LRT Ampang,LRT
4,rail_mrt_kajang,MRT Kajang,MRT
5,rail_lrt_kj,LRT Kelana Jaya,LRT
6,rail_monorail,KL Monorail,Monorail
7,rail_mrt_pjy,MRT Putrajaya,MRT
8,rail_lrt_shah_alam,LRT Shah Alam,LRT
9,rail_ets,KTMB ETS,ETS


## Results
Run the same entry point as the command-line pipeline. Outputs are reproducible; the raw file is checked for changes.

In [3]:
from src.data_cleaning import run_pipeline
clean, long = run_pipeline()
display(long[long.ridership.notna()].head(5))

{
  "source": "https://data.gov.my/data-catalogue/ridership_headline",
  "raw_sha256": "379e9b3f9bacaa0cadb9952fd654f7cdd1c68f235b189e0cf2855853adefb8bc",
  "raw_rows": 2800,
  "clean_rows": 2800,
  "long_rows": 39200,
  "date_min": "2019-01-01",
  "date_max": "2026-08-31",
  "date_gaps": 0,
  "exact_duplicates_removed": 0,
  "observed_cells": 27007,
  "ridership_sum": 2171340402,
  "policy": "Nulls and zeros preserved; conflicts, invalid dates/counts and schema changes rejected."
}


,date,service_code,ridership,service,mode,...,iso_week,day,day_of_week,day_of_week_number,is_weekend
1096,2022-01-01,bus_rkl,84122,Rapid Bus KL,Bus,...,52,1,Saturday,5,True
1097,2022-01-02,bus_rkl,77263,Rapid Bus KL,Bus,...,52,2,Sunday,6,True
1098,2022-01-03,bus_rkl,114263,Rapid Bus KL,Bus,...,1,3,Monday,0,False
1099,2022-01-04,bus_rkl,113677,Rapid Bus KL,Bus,...,1,4,Tuesday,1,False
1100,2022-01-05,bus_rkl,116090,Rapid Bus KL,Bus,...,1,5,Wednesday,2,False


### Prove that reshaping preserved counts
The validator reconstructs the full wide table, compares null positions and counts, and reconciles daily totals. A grand total alone would miss offsetting errors.

In [4]:
validate_outputs(clean, long)
wide_total = clean[list(SERVICES)].sum().sum()
long_total = long.ridership.sum()
display(pd.Series({'wide_total': wide_total, 'long_total': long_total, 'observed_cells': long.ridership.count(), 'unreported_cells': long.ridership.isna().sum(), 'duplicate_keys': long.duplicated(['date', 'service_code']).sum()}).to_frame('value'))
assert wide_total == long_total

,value
wide_total,2171340402
long_total,2171340402
observed_cells,27007
unreported_cells,12193
duplicate_keys,0


### Comparison features retain unavailable values
Monthly growth requires every day in both calendar months. Initial months and zero comparison totals stay unavailable.

In [5]:
from src.features import monthly_service_metrics
monthly = monthly_service_metrics(long)
display(monthly[monthly.service_code.eq('rail_lrt_shah_alam') & monthly.ridership.notna()][['year_month', 'ridership', 'observed_days', 'complete', 'mom_pct', 'yoy_pct']].round(2))

,year_month,ridership,observed_days,complete,mom_pct,yoy_pct
915,2026-04,0,30,True,<NA>,<NA>
917,2026-06,113575,2,False,<NA>,<NA>
918,2026-07,1990198,31,True,<NA>,<NA>
919,2026-08,1180138,31,True,-40.7,<NA>


## Takeaways
Use `python -m src.data_cleaning` to reproduce all three processed datasets. Tableau receives calendar/service fields and observation flags, leaving growth and share calculations to the workbook. Do not edit the exports by hand.